In [ ]:
# ────────────────────────────────────────────────────────────
# FULL RUN: Parse *every* TTL file under OfficeGraph → all_triples.parquet
# ────────────────────────────────────────────────────────────

import os
import glob
import rdflib
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from tqdm import tqdm
from utils import list_all_ttl_files, infer_floor_from_uri
# ─────────── CONFIGURATION ───────────
BASE_DIR       = "/Users/andreimaria/Master/Master-EDA/OfficeGraph"
SKIP_DIRS      = {"cache", "results", "cache'"}
TOP_TTLS       = ["devices_in_rooms_enrichment.ttl", "void.ttl", "wikidata_days_enrichment.ttl"]

OUT_PARQUET    = "all_triples.parquet"   # ← final output containing all triples
CHUNK_SIZE     = 1_000_000                 # ← number of triples/buffer before writing to disk
                                         #     (tune to your RAM: 500k → ~30ms to convert + ~100ms to write)
                                         #     and roughly 100 MB per 500k rows in the Parquet.



all_ttls = list_all_ttl_files(BASE_DIR)
print(f"[FULL RUN] Found {len(all_ttls)} TTL files to process.\n")

# ─────────── STEP 1: Remove any existing Parquet ───────────
if os.path.exists(OUT_PARQUET):
    print(f"[FULL RUN] Removing existing {OUT_PARQUET}")
    os.remove(OUT_PARQUET)

# ─────────── STEP 2: Define Parquet schema ───────────
schema = pa.schema([
    ("subject",   pa.string()),
    ("predicate", pa.string()),
    ("object",    pa.string()),
])

# ─────────── STEP 3: Initialize ParquetWriter to None ───────────
writer = None
total_written = 0

# ─────────── STEP 4: Iterate over all TTLs, chunk + write ───────────
for filepath in tqdm(all_ttls, desc="[FULL RUN] Processing TTLs"):
    g = rdflib.Graph()
    try:
        g.parse(filepath, format="ttl")
    except Exception as e:
        print(f"[FULL RUN] ⚠️ Failed to parse {filepath}: {e}")
        continue

    buffer = []
    for s, p, o in g:
        buffer.append({
            "subject":   str(s),
            "predicate": str(p),
            "object":    str(o)
        })

        if len(buffer) >= CHUNK_SIZE:
            # Convert buffer → DataFrame → PyArrow Table
            df_chunk = pd.DataFrame(buffer)
            table    = pa.Table.from_pandas(df_chunk, schema=schema)

            if writer is None:
                writer = pq.ParquetWriter(
                    OUT_PARQUET,
                    schema,
                    compression="snappy",
                    use_dictionary=True
                )
                print(f"[FULL RUN] → Created {OUT_PARQUET} (first {len(buffer):,} rows).")
            writer.write_table(table)
            total_written += len(buffer)
            buffer.clear()

    # Flush any remaining triples (less than CHUNK_SIZE) for this TTL
    if buffer:
        df_chunk = pd.DataFrame(buffer)
        table    = pa.Table.from_pandas(df_chunk, schema=schema)

        if writer is None:
            writer = pq.ParquetWriter(
                OUT_PARQUET,
                schema,
                compression="snappy",
                use_dictionary=True
            )
            print(f"[FULL RUN] → Created {OUT_PARQUET} (first {len(buffer):,} rows).")
        else:
            writer.write_table(table)
        total_written += len(buffer)
        buffer.clear()

# Close writer if opened
if writer is not None:
    writer.close()

print(f"\n[FULL RUN]  Finished writing ~{total_written:,} triples into {OUT_PARQUET}.")


# ─────────── STEP 5: Validate by reading back a few rows ───────────
try:
    df_preview = pd.read_parquet(OUT_PARQUET, columns=["subject", "predicate", "object"])
    print("\n[FULL RUN] Sample of first 10 rows in all_triples.parquet:\n")
    display(df_preview.head(10))
except Exception as e:
    print(f"[FULL RUN]  Could not read back {OUT_PARQUET}: {e}")


[FULL RUN] Found 447 TTL files to process.



[FULL RUN] Processing TTLs:   0%|          | 1/447 [00:04<31:18,  4.21s/it]

[FULL RUN] → Created all_triples.parquet (first 283,368 rows).


[FULL RUN] Processing TTLs:   6%|▋         | 29/447 [02:14<32:49,  4.71s/it]

In [2]:
import re
import pandas as pd

# ──────────────────────────────────────────────────
# 6a) Read the raw triples and extract device→room and room→floor
# ──────────────────────────────────────────────────
 
# Read only the three relevant predicates from “all_triples.parquet”:
print("[INFO] Reading all_triples.parquet with only subject, predicate, object columns...")
df = pd.read_parquet("all_triples.parquet", columns=["subject","predicate","object"])
print(f"[INFO] Total rows in all_triples.parquet: {len(df):,}")
# (a) device → makesMeasurement → measurement_node  (we only care later for flattening)
#     ──────────────────────────────────────────────
mm = (
    df[df["predicate"] == "https://saref.etsi.org/core/makesMeasurement"]
    .rename(columns={"subject":"device","object":"measurement_node"})[["device","measurement_node"]]
)
# (b) measurement_node → relatesToProperty → property  (again only for later)
rp = (
    df[df["predicate"] == "https://saref.etsi.org/core/relatesToProperty"]
    .rename(columns={"subject":"measurement_node","object":"property"})[["measurement_node","property"]]
)
# (c) measurement_node → hasValue → value
hv = (
    df[df["predicate"] == "https://saref.etsi.org/core/hasValue"]
    .rename(columns={"subject":"measurement_node","object":"value"})[["measurement_node","value"]]
)
# (d) measurement_node → hasTimestamp → timestamp
ht = (
    df[df["predicate"] == "https://saref.etsi.org/core/hasTimestamp"]
    .rename(columns={"subject":"measurement_node","object":"timestamp"})[["measurement_node","timestamp"]]
)

# Now extract the “device → room” and “room → raw_floor” links:
# (e) device → isContainedIn → room
d2r = (
    df[df["predicate"] == "https://saref.etsi.org/saref4bldg/isContainedIn"]
    .rename(columns={"subject":"device","object":"room"})[["device","room"]]
)

# (f) room → isSpaceOf → raw_floor
r2f = (
    df[df["predicate"] == "https://saref.etsi.org/saref4bldg/isSpaceOf"]
    .rename(columns={"subject":"room","object":"raw_floor"})[["room","raw_floor"]]
)

# Merge them to get every “device → room → raw_floor” triple:
df_raw = d2r.merge(r2f, on="room", how="left").drop_duplicates()
print(f"[INFO] Raw device→room→raw_floor rows: {len(df_raw):,}")

# ──────────────────────────────────────────────────
# 6b) Parse “raw_floor” or “room” to get clean_floor ∈ {0,1,2,…,7}
# ──────────────────────────────────────────────────

def infer_floor_from_uri(row):
    """
    Take one row with fields (room, raw_floor). Return an int clean_floor:
      - If raw_floor already looks like “https://…/VL_floor_{n}”, extract n.
      - Else, try to parse n out of the room’s URI (e.g. “zone_VL-F3-…” or “zone_Verdieping_3”).
      - If neither yields a positive integer 1..7, return 0 (meaning “unknown/no floor”).
    """

    # 1) If raw_floor itself literally contains “VL_floor_{n}”, grab n
    rf = row["raw_floor"]
    if isinstance(rf, str) and ("VL_floor_" in rf):
        # e.g. “https://…/VL_floor_3” → floor = 3
        match = re.search(r"VL_floor_(\d+)", rf)
        if match:
            return int(match.group(1))

    # 2) Otherwise, look at the room URI to see if it contains “VL-F{n}” or “Verdieping_{n}”
    room_uri = row["room"]
    if isinstance(room_uri, str):
        # a) “…/zone_VL-F5-Oost-…”, “zone_VL-F3-…”  → floor = 5 or 3
        m1 = re.search(r"zone_VL[-_]F(\d+)", room_uri)
        if m1:
            return int(m1.group(1))
        # b) “…/zone_Verdieping_3”, “zone_Verdieping_7_West…” → floor = 3 or 7
        m2 = re.search(r"zone_Verdieping_(\d+)", room_uri)
        if m2:
            return int(m2.group(1))

    # 3) If nothing matched, assign 0 (“unknown / no floor”)
    return 0

# Apply the inference, producing exactly one integer per row:
df_raw["clean_floor"] = df_raw.apply(infer_floor_from_uri, axis=1)

# Count how many rows fell into each clean_floor:
print("\n[INFO] Row counts by clean_floor (0 = unknown, 1..7 = parsed):")
print(df_raw["clean_floor"].value_counts(dropna=False))

# ──────────────────────────────────────────────────
# 6c) Drop nothing yet, but gather stats on how many are truly “unknown” (clean_floor == 0)
# ──────────────────────────────────────────────────

total_rows       = len(df_raw)
unknown_rows     = (df_raw["clean_floor"] == 0).sum()
print(f"\n[INFO] Rows with clean_floor == 0 (truly no numeric floor): {unknown_rows:,}")

# (Optional) Look at the distinct “room” URIs that ended up with floor == 0:
rooms_no_floor = df_raw.loc[df_raw["clean_floor"] == 0, "room"].unique()
print(f"[INFO] Distinct rooms missing a numeric floor: {len(rooms_no_floor)}")
print("  Sample:", list(rooms_no_floor)[:10])

# Because we do *not* want to drop entire devices, we will keep all rows,
# assigning clean_floor=0 where necessary.  
#
# If later you decide that some of those “room_urn-…” or “zone_Kantine” URIs
# actually belong on floor X, you can insert them here as overrides. E.g.:
#
#   OVERRIDES = {
#     "https://interconnectproject.eu/example/zone_Kantine": 1,
#     "https://interconnectproject.eu/example/room_urn-…": 4,
#   }
#   df_raw.loc[df_raw["room"].isin(OVERRIDES), "clean_floor"] = \
#       df_raw["room"].map(OVERRIDES)

# ──────────────────────────────────────────────────
# 6d) Collapse “multi‐floor” devices: pick the *mode* of clean_floor for each device
# ──────────────────────────────────────────────────

# First see how many devices appear under more than one nonzero floor:
device_floor_counts = (
    df_raw[df_raw["clean_floor"] != 0]
    .groupby("device")["clean_floor"]
    .nunique()
    .reset_index(name="num_distinct_floors")
)
multi_floor_devices = device_floor_counts[device_floor_counts["num_distinct_floors"] > 1]
print(f"\n[INFO] Devices mapping to >1 nonzero floors: {len(multi_floor_devices)}")
print("  Sample:", list(multi_floor_devices["device"][:5]))




# Build a little table of (device → resolved_floor):
resolved = (
    df_raw.groupby("device")
    .apply(mode_floor_for_device)
    .reset_index(name="chosen_floor")
)

# Now, “resolved” has exactly one row per device:
print(f"\n[INFO] After mode‐selection, {len(resolved)} devices → one chosen_floor each.")


# ──────────────────────────────────────────────────
# 6e) Join back to pick exactly one row per device, but keep “room” for reference.
#     (If desired, you could record the *primary* room, or just leave it blank.)
# ──────────────────────────────────────────────────

# (Option A) If you want to store which *room* ended up supplying that mode_floor,
# you could do a merge that keeps the first‐appearing row for that (device,chosen_floor).
# However, typically the “room” is less important than knowing “device → floor.”
# For simplicity, let’s just choose the first matching row in df_raw:
merged = (
    df_raw.merge(resolved, on="device", how="inner")
    .query("clean_floor == chosen_floor")    # keep only the row(s) where clean_floor matches chosen_floor
    .sort_values(["device", "room"])         # sort so that we can pick “.drop_duplicates(keep='first')”
    .drop_duplicates(subset=["device"], keep="first")
    .reset_index(drop=True)
)

# If you prefer to keep **all rooms** for that device that share the same chosen_floor,
# you could simply _not_ drop_duplicates. But usually we only need one “representative” room.

print(f"[INFO] Final device→room→clean_floor rows: {len(merged):,}")
print(f"[INFO] Unique devices now: {merged['device'].nunique()}")
print(f"[INFO] Unique rooms now:   {merged['room'].nunique()}")
print(f"[INFO] Unique floors now:  {merged['clean_floor'].nunique()}")
print(f"\n[INFO] Sample of final (device,room,clean_floor):")
print(merged[["device","room","clean_floor"]].head(10))

# ──────────────────────────────────────────────────
# 6f) Save the cleaned mapping for downstream use
# ──────────────────────────────────────────────────

# We’ll write out “device_room_floor_cleaned.csv” with exactly one row per device:
merged.rename(columns={"clean_floor":"floor"}).to_csv(
    "device_room_floor_cleaned.csv",
    index=False
)

print("\n[INFO] Saved cleaned device→floor (including unknown floor=0) as device_room_floor_cleaned.csv")

[INFO] Reading all_triples.parquet with only subject, predicate, object columns...
[INFO] Total rows in all_triples.parquet: 89,324,854
[INFO] Raw device→room→raw_floor rows: 2,701

[INFO] Row counts by clean_floor (0 = unknown, 1..7 = parsed):
clean_floor
2    454
3    450
7    435
4    394
1    386
6    212
0    203
5    167
Name: count, dtype: int64

[INFO] Rows with clean_floor == 0 (truly no numeric floor): 203
[INFO] Distinct rooms missing a numeric floor: 68
  Sample: ['https://interconnectproject.eu/example/room_urn-Room-SmartThings-2fbeed1f-105f-4a11-93b8-f19ef88167a6', 'https://interconnectproject.eu/example/room_urn-Room-SmartThings-ed73d598-b9ed-4448-9ece-e553e4b23e32', 'https://interconnectproject.eu/example/room_urn-Room-SmartThings-d84929fa-b572-4fb2-aa8e-c8be86f38a4b', 'https://interconnectproject.eu/example/room_urn-Room-SmartThings-60413db4-8641-4b17-bb6d-3b0ed4fada60', 'https://interconnectproject.eu/example/room_urn-Room-SmartThings-0c9f3381-ff4c-4b93-a5d7-ad95ee322

NameError: name 'mode_floor_for_device' is not defined

In [ ]:
# ────────────────────────────────────────────────────────────
# UPDATED STEP 7: Build “flattened measurements” using cleaned mapping
# ────────────────────────────────────────────────────────────

import pandas as pd

# (a) Read our “all_triples.parquet” again (only the three columns we need)
df_triples = pd.read_parquet("all_triples.parquet", columns=["subject", "predicate", "object"])

# (b) Extract device → makesMeasurement → measurement_node
mm = (
    df_triples[df_triples["predicate"] == "https://saref.etsi.org/core/makesMeasurement"]
    .rename(columns={"subject": "device", "object": "measurement_node"})
)[["device", "measurement_node"]]

# (c) Extract measurement_node → relatesToProperty → property
rp = (
    df_triples[df_triples["predicate"] == "https://saref.etsi.org/core/relatesToProperty"]
    .rename(columns={"subject": "measurement_node", "object": "property"})
)[["measurement_node", "property"]]

# (d) Extract measurement_node → hasValue → value
hv = (
    df_triples[df_triples["predicate"] == "https://saref.etsi.org/core/hasValue"]
    .rename(columns={"subject": "measurement_node", "object": "value"})
)[["measurement_node", "value"]]

# (e) Extract measurement_node → hasTimestamp → timestamp
ht = (
    df_triples[df_triples["predicate"] == "https://saref.etsi.org/core/hasTimestamp"]
    .rename(columns={"subject": "measurement_node", "object": "timestamp"})
)[["measurement_node", "timestamp"]]

# (f) Merge (a)+(b)+(c)+(d) → one row per measurement event
tmp = (
    mm
    .merge(rp, on="measurement_node", how="inner")
    .merge(hv, on="measurement_node", how="inner")
    .merge(ht, on="measurement_node", how="inner")
)

# (g) Read the **cleaned** device→floor mapping
df_dev_room_floor = pd.read_csv("device_room_floor_cleaned.csv")
#   ├─ columns: [device, room, room_type, clean_floor]

# (h) Merge flattened measurements with device→floor info
measurements = tmp.merge(
    df_dev_room_floor[["device","room", "floor"]], 
    on="device", 
    how="left"
)

# (i) Convert “value” → float, “timestamp” → datetime
measurements["value"]     = measurements["value"].astype(float)
measurements["timestamp"] = pd.to_datetime(measurements["timestamp"])

# (j) Write out the final flattened table
measurements.to_parquet("flattened_measurements.parquet", index=False)

print(f"[UPDATED STEP 7] Flattened measurements rows: {len(measurements)}")
print(f"[UPDATED STEP 7] NaN floors in measurements: {measurements['floor'].isna().sum()}")

# If any NaN floors remain, you can drop those rows or inspect them separately.

[UPDATED STEP 7] Flattened measurements rows: 14883252
[UPDATED STEP 7] NaN floors in measurements: 137369


In [ ]:
''# ───────────────────────────────────────────────────────────────────────────
# EXTRA CHECKS: NaN‐floor rooms and multi‐floor device timelines
# ───────────────────────────────────────────────────────────────────────────

import pandas as pd
import matplotlib.pyplot as plt

# (A) Load the flattened measurements with room & floor columns
fm = pd.read_parquet("flattened_measurements.parquet")

# (B) Identify rows where floor is NaN
nan_floor_rows = fm[fm["floor"].isna()]

print(f"[CHECK] {len(nan_floor_rows)} measurement rows have floor = NaN.")
print("\nSample of rooms missing a floor (up to 10 distinct rooms):")
for room_uri in nan_floor_rows["room"].unique()[:10]:
    print("  ", room_uri)
print("\n[CHECK] Total distinct rooms with NaN‐floor:", nan_floor_rows["room"].nunique())

# (C) Attempt to recover floor for those rooms by querying any isSpaceOf triple they might have
#     (i.e. look up in all_triples.parquet for those room URIs → floor, even if not 'VL_floor_*')

# 1) Read only the room/isSpaceOf/floor triples from all_triples.parquet
triples = pd.read_parquet("all_triples.parquet", columns=["subject", "predicate", "object"])
r2f_full = triples[triples["predicate"] == "https://saref.etsi.org/saref4bldg/isSpaceOf"] \
               .rename(columns={"subject": "room", "object": "floor"})[["room", "floor"]] \
               .drop_duplicates()

# 2) For each NaN‐floor room, see if it appears in r2f_full with any floor
print("\n[CHECK] Attempting to find any floor mapping for NaN‐floor rooms in full isSpaceOf:")
rooms_with_nan = nan_floor_rows["room"].unique()
recovered = r2f_full[r2f_full["room"].isin(rooms_with_nan)]
if not recovered.empty:
    print(f"  → {len(recovered)} (room, floor) pairs found in full isSpaceOf for these rooms:")
    display(recovered.head(10))
else:
    print("  → No matching isSpaceOf triples found for these rooms.")

# (D) Plot timestamp vs. floor for the 3 multi‐floor devices to see if/when they moved
multi_devices = ["https://interconnectproject.eu/example/R5_28",
                 "https://interconnectproject.eu/example/R5_120",
                 "https://interconnectproject.eu/example/R5_8"]

for dev in multi_devices:
    dev_data = fm[fm["device"] == dev].sort_values("timestamp")
    if dev_data.empty:
        print(f"\n[CHECK] No measurements found for {dev}")
        continue

''' plt.figure(figsize=(8, 3))
    # Convert floor URIs into simple labels (e.g., "Floor 3", "Floor 5")
    floor_labels = dev_data["floor"].fillna("Unknown").apply(lambda uri: uri.split("_")[-1] if pd.notna(uri) and "VL_floor" in uri else "Unknown")
    # Plot: timestamp on x, numeric code per floor label on y
    unique_floors = sorted(floor_labels.unique())
    y_mapping = {fl: idx for idx, fl in enumerate(unique_floors)}
    y_values = floor_labels.map(y_mapping)

    plt.scatter(dev_data["timestamp"], y_values, s=4)
    plt.yticks(list(y_mapping.values()), list(y_mapping.keys()))
    plt.title(f"Device {dev.split('/')[-1]}: Floor assignment over time")
    plt.xlabel("Timestamp")
    plt.ylabel("Floor")
    plt.tight_layout()
    plt.show()
'''
# (E) After inspection, if you decide to drop NaN‐floor rows and multi‐floor devices,
#     uncomment below lines to remove them from device_room_floor.csv and re‐save:

# device_room_floor = pd.read_csv("device_room_floor.csv")
# # Drop NaN‐floor rows:
# device_room_floor = device_room_floor.dropna(subset=["floor"])
# # Drop multi‐floor devices entirely:
# to_drop = multi_devices
# device_room_floor = device_room_floor[~device_room_floor["device"].isin(to_drop)]
# # Re‐save:
# device_room_floor.to_csv("device_room_floor_cleaned.csv", index=False)
# print("\n[INFO] Dropped NaN‐floor rows and multi‐floor devices; saved to device_room_floor_cleaned.csv")''

[CHECK] 137369 measurement rows have floor = NaN.

Sample of rooms missing a floor (up to 10 distinct rooms):
   None

[CHECK] Total distinct rooms with NaN‐floor: 0

[CHECK] Attempting to find any floor mapping for NaN‐floor rooms in full isSpaceOf:
  → No matching isSpaceOf triples found for these rooms.


' plt.figure(figsize=(8, 3))\n    # Convert floor URIs into simple labels (e.g., "Floor 3", "Floor 5")\n    floor_labels = dev_data["floor"].fillna("Unknown").apply(lambda uri: uri.split("_")[-1] if pd.notna(uri) and "VL_floor" in uri else "Unknown")\n    # Plot: timestamp on x, numeric code per floor label on y\n    unique_floors = sorted(floor_labels.unique())\n    y_mapping = {fl: idx for idx, fl in enumerate(unique_floors)}\n    y_values = floor_labels.map(y_mapping)\n\n    plt.scatter(dev_data["timestamp"], y_values, s=4)\n    plt.yticks(list(y_mapping.values()), list(y_mapping.keys()))\n    plt.title(f"Device {dev.split(\'/\')[-1]}: Floor assignment over time")\n    plt.xlabel("Timestamp")\n    plt.ylabel("Floor")\n    plt.tight_layout()\n    plt.show()\n'

In [ ]:
measurements

,device,measurement_node,property,value,timestamp,room,floor
0,https://interconnectproject.eu/example/R5_15,https://interconnectproject.eu/example/measure...,https://interconnectproject.eu/example/propert...,21.9,2022-12-02 20:17:00,https://interconnectproject.eu/example/roomnam...,1.0
1,https://interconnectproject.eu/example/R5_15,https://interconnectproject.eu/example/measure...,https://interconnectproject.eu/example/propert...,21.6,2023-01-21 17:44:00,https://interconnectproject.eu/example/roomnam...,1.0
2,https://interconnectproject.eu/example/R5_15,https://interconnectproject.eu/example/measure...,https://interconnectproject.eu/example/propert...,22.6,2022-11-24 10:38:00,https://interconnectproject.eu/example/roomnam...,1.0
3,https://interconnectproject.eu/example/R5_15,https://interconnectproject.eu/example/measure...,https://interconnectproject.eu/example/propert...,459.0,2022-06-28 00:00:00,https://interconnectproject.eu/example/roomnam...,1.0
4,https://interconnectproject.eu/example/R5_15,https://interconnectproject.eu/example/measure...,https://interconnectproject.eu/example/propert...,23.6,2022-06-14 17:21:00,https://interconnectproject.eu/example/roomnam...,1.0
...,...,...,...,...,...,...,...
14883247,https://interconnectproject.eu/example/SmartSe...,https://interconnectproject.eu/example/measure...,https://interconnectproject.eu/example/propert...,21.1,2022-03-02 17:20:00,https://interconnectproject.eu/example/room_ur...,0.0
14883248,https://interconnectproject.eu/example/SmartSe...,https://interconnectproject.eu/example/measure...,https://interconnectproject.eu/example/propert...,24.9,2022-03-19 16:05:00,https://interconnectproject.eu/example/room_ur...,0.0
14883249,https://interconnectproject.eu/example/SmartSe...,https://interconnectproject.eu/example/measure...,https://interconnectproject.eu/example/propert...,11.8,2022-03-25 08:32:00,https://interconnectproject.eu/example/room_ur...,0.0
14883250,https://interconnectproject.eu/example/SmartSe...,https://interconnectproject.eu/example/measure...,https://interconnectproject.eu/example/propert...,1.0,2022-03-05 23:00:00,https://interconnectproject.eu/example/room_ur...,0.0


In [ ]:
types = df[df["predicate"] == "http://www.w3.org/1999/02/22-rdf-syntax-ns#type"]["object"].unique()
pd.Series(types).to_csv("distinct_rdf_types.txt", index=False)


In [ ]:
types

array(['https://saref.etsi.org/core/Measurement',
       'https://interconnectproject.eu/example/CO2Level',
       'https://saref.etsi.org/core/Temperature',
       'https://saref.etsi.org/saref4ener/Device',
       'https://saref.etsi.org/core/Humidity',
       'https://interconnectproject.eu/example/RunningTime',
       'https://saref.etsi.org/core/Occupancy',
       'https://saref.etsi.org/saref4bldg/BuildingSpace',
       'https://saref.etsi.org/saref4bldg/Building',
       'https://interconnectproject.eu/example/Contact',
       'https://interconnectproject.eu/example/BatteryLevel',
       'https://interconnectproject.eu/example/DeviceStatus',
       'https://saref.etsi.org/core/Power',
       'https://saref.etsi.org/core/Motion',
       'https://interconnectproject.eu/example/thermostatHeatingSetpoint',
       'http://purl.org/dc/terms/PeriodOfTime',
       'http://www.w3.org/ns/dcat#Catalog',
       'http://rdfs.org/ns/void#Dataset',
       'http://xmlns.com/foaf/0.1/Person',
  